# sqlite-online-jit Matrix Analysis

This notebook visualizes matrix benchmark outputs generated by `sqlite-online-jit` with:
- backend: `rawdog-jit`, `llvm-jit`
- hash: curated good-hash set
- requested vector widths: `1`, `2`, `4`, `8`, `16`
- repeated table builds per permutation (`--build-runs`)

It focuses on both query-time speedup and build-time cost (including `PhOnlineJitCreateTable32` timing).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
EXAMPLE_ROOT = Path('..').resolve()
RESULTS_DIR = EXAMPLE_ROOT / 'results' / 'latest'
SUMMARY_CSV = RESULTS_DIR / 'summary.csv'
DETAILED_CSV = RESULTS_DIR / 'detailed.csv'

print('Summary CSV :', SUMMARY_CSV)
print('Detailed CSV:', DETAILED_CSV)


In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
detailed = pd.read_csv(DETAILED_CSV)
summary_ok = summary[summary['outcome'] == 'ok'].copy()
detailed_ok = detailed[(detailed['create_ok'] == 1) & (detailed['query_ok'] == 1)].copy()

print('summary rows:', len(summary), 'ok:', len(summary_ok))
print('detailed rows:', len(detailed), 'ok:', len(detailed_ok))
summary_ok.head(3)


In [ ]:
baseline_ms = float(summary_ok['baseline_avg_ms'].iloc[0])
print('baseline avg ms:', baseline_ms)
summary_ok[['query_speedup', 'end_to_end_speedup', 'break_even_queries', 'create_avg_ms', 'table_create_avg_ms', 'compile_avg_ms']].describe()


In [ ]:
def plot_heatmap(metric: str, title: str, fmt: str = '.2f', cmap: str = 'viridis'):
    backends = ['rawdog-jit', 'llvm-jit']
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

    for ax, backend in zip(axes, backends):
        subset = summary_ok[summary_ok['backend'] == backend]
        pivot = subset.pivot_table(index='hash', columns='req_vec', values=metric, aggfunc='mean')
        pivot = pivot[[1, 2, 4, 8, 16]]
        sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap, ax=ax)
        ax.set_title(f'{backend} - {metric}')
        ax.set_xlabel('Requested Vector Width')
        ax.set_ylabel('Hash Function')

    fig.suptitle(title, y=1.02)
    plt.tight_layout()


In [ ]:
plot_heatmap('query_speedup', 'Query-Only Speedup by Hash/Vector/Backend', fmt='.3f', cmap='mako')
plot_heatmap('end_to_end_speedup', 'End-to-End Speedup (Create + Query)', fmt='.3f', cmap='crest')
plot_heatmap('table_create_avg_ms', 'Table Creation Cost (PhOnlineJitCreateTable32 ms)', fmt='.2f', cmap='rocket_r')
plot_heatmap('compile_avg_ms', 'Compile Cost (ms)', fmt='.2f', cmap='flare_r')


In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=detailed_ok, x='req_vec', y='table_create_ms', hue='backend')
plt.title('Table-Create Distribution Across Repeated Runs')
plt.xlabel('Requested Vector Width')
plt.ylabel('PhOnlineJitCreateTable32 Time (ms)')
plt.tight_layout()


In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=summary_ok,
    x='table_create_avg_ms',
    y='query_speedup',
    hue='backend',
    style='req_vec',
    s=90,
)
plt.title('Query Speedup vs Table-Create Cost')
plt.xlabel('Table-Create Avg (ms)')
plt.ylabel('Query Speedup (baseline/query)')
plt.tight_layout()


In [ ]:
display_cols = [
    'backend', 'hash', 'req_vec',
    'query_speedup', 'end_to_end_speedup', 'break_even_queries',
    'create_avg_ms', 'table_create_avg_ms', 'compile_avg_ms',
    'jit_success_runs', 'jit_fallback_runs'
]

print('Top 15 query-speedup permutations')
summary_ok.sort_values('query_speedup', ascending=False)[display_cols].head(15)


In [ ]:
print('Top 15 end-to-end permutations')
summary_ok.sort_values('end_to_end_speedup', ascending=False)[display_cols].head(15)
